# Evolution of the LSTM temperature-forecasting models

This notebook reconstructs and explains the four supplied experiment summaries. Each section contains a complete Keras builder that can be run independently after the shared setup cell.

The saved summaries reveal four distinct architectures. The **2026-07-01** model is a fixed-horizon stacked-LSTM regressor, while **2026-07-23** is a 72-to-24-hour residual forecaster built around the previous day's temperature curve.

| Experiment | Input per sample | Prediction | Main design | Parameters |
|---|---:|---:|---|---:|
| 2026-07-01 | 24 hours × 10 features | one temperature, 3 hours ahead | Multi-Horizontal Stacked LSTM | 33,238 |
| 2026-07-23 | 72 hours × 11 features | temperatures for hours 1–24 | LSTM correction to previous-day baseline | 20,287 |
| 2026-07-27 | 72 hours × 11 features | temperatures for hours 1–24 | CNN + LSTM + self-attention + previous-day residual | 212,232 |
| 2026-08-02 | 72 hours × 11 features | temperatures for hours 1–24 | compact encoder + self/cross-attention + learned baseline blend | 100,195 |

`None` in a model summary is the batch dimension. For example, `(None, 72, 11)` means any batch size, 72 hourly timesteps, and 11 features per hour.


## 1. Shared setup and feature contracts

The early model does **not** receive past temperature as an input feature; temperature is only the target. The later 24-hour models add past temperature to the feature tensor, which enables previous-day, persistence, and trend baselines without target leakage—only observations available before the forecast origin are used.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, regularizers
from tensorflow.keras.models import Model

tf.keras.utils.set_random_seed(21)

EARLY_FEATURE_COLUMNS = [
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
    "is_day",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

FEATURE_COLUMNS = [
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "wind_speed_10m",
    "cloud_cover",
    "precipitation",
    "is_day",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
]

TARGET_COLUMN = "temperature_2m"
INPUT_HOURS = 72
OUTPUT_HOURS = 24

TEMPERATURE_FEATURE_INDEX = FEATURE_COLUMNS.index(TARGET_COLUMN)
HOUR_SIN_INDEX = FEATURE_COLUMNS.index("hour_sin")
HOUR_COS_INDEX = FEATURE_COLUMNS.index("hour_cos")
DAY_SIN_INDEX = FEATURE_COLUMNS.index("day_sin")
DAY_COS_INDEX = FEATURE_COLUMNS.index("day_cos")


def validate_training_tensor(X_train, hours, feature_count):
    '''Fail early when a builder receives the wrong tensor contract.'''
    if not isinstance(X_train, np.ndarray) or X_train.ndim != 3:
        raise ValueError("X_train must be a 3-D NumPy array: samples × hours × features.")
    if X_train.shape[1:] != (hours, feature_count):
        raise ValueError(
            f"Expected (*, {hours}, {feature_count}); received {X_train.shape}."
        )
    if not np.isfinite(X_train).all():
        raise ValueError("X_train contains NaN or infinite values.")


## 2. Standard Stacked LSTM and Residual Baseline

### 2026-07-01: Standard Stacked LSTM regression

This model consumes 24 hours of 10 non-temperature weather and calendar features and predicts one temperature 3 hours after the end of the window. It uses 64- and 32-unit LSTMs followed by 32- and 16-unit dense layers. Ordinary dropout is applied between stages. The model has 33,238 parameters, including 21 non-trainable normalization statistics.

### 2026-07-23: Residual Baseline

This model changes both the input and the forecast task. It consumes 72 hours of 11 features—including observed temperature—and predicts all 24 temperatures for the next day. A 48/24-unit LSTM encoder predicts a 24-value adjustment vector. That adjustment is added to the final 24 observed temperatures, which represent the same hours on the previous day. The network therefore learns the departure from a strong daily-cycle baseline instead of rebuilding the entire temperature curve from zero.

The residual model has 20,287 parameters and uses layer normalization, L2 regularization, dropout, gradient clipping, and Huber loss.


In [2]:
def build_model_2026_07_01(X_train):
    '''Builder for experiment 2026-07-01_17-14-36.'''
    validate_training_tensor(X_train, hours=24, feature_count=10)

    normalizer = layers.Normalization(axis=-1)
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(24, 10),
        name="weather_sequence",
    )
    x = layers.TimeDistributed(
        normalizer,
        name="time_distributed",
    )(inputs)

    x = layers.LSTM(
        64,
        return_sequences=True,
        name="lstm",
    )(x)
    x = layers.Dropout(0.20, name="dropout")(x)
    x = layers.LSTM(32, name="lstm_1")(x)
    x = layers.Dropout(0.20, name="dropout_1")(x)

    x = layers.Dense(32, activation="relu", name="dense")(x)
    x = layers.Dropout(0.20, name="dropout_2")(x)
    x = layers.Dense(16, activation="relu", name="dense_1")(x)
    outputs = layers.Dense(1, name="temperature_prediction")(x)

    model = Model(inputs=inputs, outputs=outputs, name="functional")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


def build_model_2026_07_23(X_train, learning_rate=3e-4):
    '''Builder for experiment 2026-07-23_19-38-36 (Residual Baseline).'''
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(
        axis=-1,
        name="feature_normalization",
    )
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    x = layers.LSTM(
        48,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(1e-4),
        name="lstm_1",
    )(x)
    x = layers.LayerNormalization(name="lstm_1_normalization")(x)

    x = layers.LSTM(
        24,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(1e-4),
        name="lstm_2",
    )(x)
    x = layers.LayerNormalization(name="lstm_2_normalization")(x)

    x = layers.Dense(
        32,
        activation="relu",
        kernel_regularizer=regularizers.l2(1e-4),
        name="dense_features",
    )(x)
    x = layers.Dropout(0.20, name="dense_dropout")(x)

    temperature_adjustment = layers.Dense(
        OUTPUT_HOURS,
        name="temperature_adjustment",
    )(x)
    previous_day_temperature = inputs[
        :,
        -OUTPUT_HOURS:,
        TEMPERATURE_FEATURE_INDEX,
    ]
    outputs = layers.Add(name="temperature_prediction")(
        [previous_day_temperature, temperature_adjustment]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_24h",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        ),
        loss=tf.keras.losses.Huber(delta=1.5),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


### What changed after the Residual Baseline?

The July 27 Attention-Augmented model retains the July 23 residual idea—previous-day temperature plus a learned adjustment—but adds convolutional feature extraction, a larger LSTM encoder, self-attention, richer history pooling, and a horizon-aware decoder. The Cross-Attention model later expands the baseline idea again by blending previous-day, persistence, and trend forecasts.


## 3. Experiment 2026-07-27: convolutional LSTM with self-attention

This model has five conceptual stages:

1. A causal convolution and a separable convolution find short local weather patterns. A residual connection protects the original convolutional representation.
2. A 96-unit LSTM models the ordered 72-hour history.
3. Four-head self-attention lets every historical hour compare itself with every other hour. A Transformer-style feed-forward block follows it.
4. Average, maximum, and latest-state pooling summarize the encoded history from complementary viewpoints.
5. A horizon-aware decoder predicts 24 corrections, which are added to the temperatures observed at the same hours on the previous day.

The learned horizon embedding is important: forecast hour 1 and forecast hour 24 should not use exactly the same decoder representation.


In [3]:
@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class PreviousDayTemperature(layers.Layer):
    '''Return the latest 24 observed temperatures as a forecast baseline.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.feature_index = int(feature_index)

    def call(self, inputs):
        return inputs[:, -self.output_hours :, self.feature_index]

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "feature_index": self.feature_index,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class HorizonEmbedding(layers.Layer):
    '''Learn one embedding vector for each future hour.'''

    def __init__(self, output_hours=OUTPUT_HOURS, embedding_dim=8, **kwargs):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.embedding_dim = int(embedding_dim)
        self.embedding = layers.Embedding(
            input_dim=self.output_hours,
            output_dim=self.embedding_dim,
        )

    def call(self, inputs):
        indices = tf.range(self.output_hours)
        embedded = self.embedding(indices)[tf.newaxis, :, :]
        return tf.tile(embedded, [tf.shape(inputs)[0], 1, 1])

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "embedding_dim": self.embedding_dim,
            }
        )
        return config


def build_model_2026_07_27(X_train, learning_rate=5e-4):
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(axis=-1, name="feature_normalization")
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    causal = layers.Conv1D(
        64,
        kernel_size=5,
        padding="causal",
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="causal_conv",
    )(x)
    local = layers.SeparableConv1D(
        64,
        kernel_size=3,
        padding="same",
        activation="swish",
        depthwise_regularizer=regularizers.l2(5e-5),
        pointwise_regularizer=regularizers.l2(5e-5),
        name="local_weather_block",
    )(causal)
    local = layers.Dropout(0.10, name="conv_dropout")(local)
    x = layers.Add(name="conv_residual_add")([causal, local])
    x = layers.LayerNormalization(name="conv_normalization")(x)

    x = layers.LSTM(
        96,
        return_sequences=True,
        dropout=0.10,
        kernel_regularizer=regularizers.l2(5e-5),
        name="lstm_encoder",
    )(x)

    attention = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=24,
        dropout=0.10,
        name="history_attention",
    )(x, x)
    x = layers.Add(name="attention_residual_add")([x, attention])
    x = layers.LayerNormalization(name="attention_normalization")(x)

    feed_forward = layers.Dense(
        192,
        activation="swish",
        name="attention_ff_1",
    )(x)
    feed_forward = layers.Dropout(
        0.10,
        name="attention_ff_dropout",
    )(feed_forward)
    feed_forward = layers.Dense(
        96,
        name="attention_ff_2",
    )(feed_forward)
    x = layers.Add(name="feed_forward_residual_add")([x, feed_forward])
    encoded = layers.LayerNormalization(
        name="encoder_output_normalization",
    )(x)

    latest = layers.Cropping1D(
        cropping=(INPUT_HOURS - 1, 0),
        name="latest_timestep_crop",
    )(encoded)
    average = layers.GlobalAveragePooling1D(
        name="average_context",
    )(encoded)
    maximum = layers.GlobalMaxPooling1D(
        name="maximum_context",
    )(encoded)
    latest = layers.Reshape((96,), name="latest_context")(latest)

    context = layers.Concatenate(name="combined_context")(
        [average, maximum, latest]
    )
    context = layers.Dense(
        160,
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="forecast_features_1",
    )(context)
    context = layers.Dropout(0.20, name="forecast_dropout_1")(context)
    context = layers.Dense(
        80,
        activation="swish",
        kernel_regularizer=regularizers.l2(5e-5),
        name="forecast_features_2",
    )(context)
    context = layers.Dropout(0.10, name="forecast_dropout_2")(context)

    previous_day = PreviousDayTemperature(
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        name="previous_day_temperature",
    )(inputs)
    repeated_context = layers.RepeatVector(
        OUTPUT_HOURS,
        name="repeat_context",
    )(context)
    horizon = HorizonEmbedding(
        output_hours=OUTPUT_HOURS,
        embedding_dim=12,
        name="horizon_embedding",
    )(context)
    previous_day_expanded = layers.Reshape(
        (OUTPUT_HOURS, 1),
        name="previous_day_expanded",
    )(previous_day)

    decoder = layers.Concatenate(name="horizon_decoder_input")(
        [repeated_context, horizon, previous_day_expanded]
    )
    decoder = layers.Dense(
        64,
        activation="swish",
        name="decoder_dense_1",
    )(decoder)
    decoder = layers.Dropout(0.10, name="decoder_dropout")(decoder)
    decoder = layers.Dense(
        32,
        activation="swish",
        name="decoder_dense_2",
    )(decoder)
    adjustment = layers.Dense(
        1,
        name="temperature_adjustment_per_hour",
    )(decoder)
    adjustment = layers.Reshape(
        (OUTPUT_HOURS,),
        name="temperature_adjustment",
    )(adjustment)
    outputs = layers.Add(name="temperature_prediction")(
        [previous_day, adjustment]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_attention_24h",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        ),
        loss=tf.keras.losses.Huber(),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


## 4. Experiment 2026-08-02: cross-attentive multi-baseline model

The final supplied architecture is smaller than the July 27 model—100,195 instead of 212,232 parameters—but introduces more forecasting structure.

- Two LSTMs and self-attention create a 40-dimensional representation for every historical hour.
- Each of the 24 future hours becomes a separate query containing its horizon embedding, future calendar phase, and three simple temperature baselines.
- Cross-attention allows each future-hour query to inspect all 72 encoded historical hours directly. The model no longer has to recover all temporal detail from one pooled vector.
- A GRU decodes the 24 queries jointly, allowing neighboring forecast hours to remain coherent.
- Softmax weights blend previous-day, persistence, and damped-trend baselines separately at every horizon. A learned correction is then added to the blend.
- `LevelChangeHuber` penalizes both temperature-level error and error in hour-to-hour changes, with modestly higher weight on later horizons.

The following helpers reproduce the custom layers used by the attached `model.py`.


In [4]:
@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class TemperatureBaselines(layers.Layer):
    '''Previous-day, persistence, and damped recent-trend forecasts.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        trend_lookback=6,
        trend_decay_hours=8.0,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.feature_index = int(feature_index)
        self.trend_lookback = int(trend_lookback)
        self.trend_decay_hours = float(trend_decay_hours)

    def call(self, inputs):
        temperature = inputs[:, :, self.feature_index]
        previous_day = temperature[:, -self.output_hours :]

        latest = temperature[:, -1:]
        persistence = tf.repeat(latest, repeats=self.output_hours, axis=1)

        earlier = temperature[
            :, -(self.trend_lookback + 1) : -self.trend_lookback
        ]
        slope_per_hour = (latest - earlier) / tf.cast(
            self.trend_lookback,
            inputs.dtype,
        )
        horizon = tf.cast(
            tf.range(1, self.output_hours + 1)[tf.newaxis, :],
            inputs.dtype,
        )
        damping = tf.exp(
            -horizon / tf.cast(self.trend_decay_hours, inputs.dtype)
        )
        trend = persistence + slope_per_hour * horizon * damping
        return tf.stack([previous_day, persistence, trend], axis=-1)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "feature_index": self.feature_index,
                "trend_lookback": self.trend_lookback,
                "trend_decay_hours": self.trend_decay_hours,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class FutureCalendarFeatures(layers.Layer):
    '''Rotate the last cyclical time features into the next 24 hours.'''

    def __init__(
        self,
        output_hours=OUTPUT_HOURS,
        hour_sin_index=HOUR_SIN_INDEX,
        hour_cos_index=HOUR_COS_INDEX,
        day_sin_index=DAY_SIN_INDEX,
        day_cos_index=DAY_COS_INDEX,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.output_hours = int(output_hours)
        self.hour_sin_index = int(hour_sin_index)
        self.hour_cos_index = int(hour_cos_index)
        self.day_sin_index = int(day_sin_index)
        self.day_cos_index = int(day_cos_index)

    def call(self, inputs):
        dtype = inputs.dtype
        horizon = tf.cast(
            tf.range(1, self.output_hours + 1)[tf.newaxis, :],
            dtype,
        )

        last_hour_sin = inputs[:, -1, self.hour_sin_index][:, tf.newaxis]
        last_hour_cos = inputs[:, -1, self.hour_cos_index][:, tf.newaxis]
        last_day_sin = inputs[:, -1, self.day_sin_index][:, tf.newaxis]
        last_day_cos = inputs[:, -1, self.day_cos_index][:, tf.newaxis]

        hour_angle = horizon * tf.cast(2.0 * np.pi / 24.0, dtype)
        hour_cos_rotation = tf.cos(hour_angle)
        hour_sin_rotation = tf.sin(hour_angle)
        future_hour_sin = (
            last_hour_sin * hour_cos_rotation
            + last_hour_cos * hour_sin_rotation
        )
        future_hour_cos = (
            last_hour_cos * hour_cos_rotation
            - last_hour_sin * hour_sin_rotation
        )

        day_angle = horizon * tf.cast(2.0 * np.pi / (24.0 * 365.25), dtype)
        day_cos_rotation = tf.cos(day_angle)
        day_sin_rotation = tf.sin(day_angle)
        future_day_sin = (
            last_day_sin * day_cos_rotation
            + last_day_cos * day_sin_rotation
        )
        future_day_cos = (
            last_day_cos * day_cos_rotation
            - last_day_sin * day_sin_rotation
        )

        return tf.stack(
            [
                future_hour_sin,
                future_hour_cos,
                future_day_sin,
                future_day_cos,
            ],
            axis=-1,
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "output_hours": self.output_hours,
                "hour_sin_index": self.hour_sin_index,
                "hour_cos_index": self.hour_cos_index,
                "day_sin_index": self.day_sin_index,
                "day_cos_index": self.day_cos_index,
            }
        )
        return config


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class WeightedBaseline(layers.Layer):
    '''Apply learned per-hour softmax weights to baseline candidates.'''

    def call(self, inputs):
        baselines, weights = inputs
        return tf.reduce_sum(baselines * weights, axis=-1)


@tf.keras.utils.register_keras_serializable(package="WeatherNotebook")
class LevelChangeHuber(tf.keras.losses.Loss):
    '''Huber loss for forecast levels plus adjacent-hour changes.'''

    def __init__(
        self,
        delta=1.0,
        change_weight=0.25,
        late_horizon_weight=0.30,
        name="level_change_huber",
        **kwargs,
    ):
        super().__init__(name=name, **kwargs)
        self.delta = float(delta)
        self.change_weight = float(change_weight)
        self.late_horizon_weight = float(late_horizon_weight)

    def _elementwise_huber(self, error):
        error = tf.convert_to_tensor(error)
        absolute_error = tf.abs(error)
        delta = tf.cast(self.delta, error.dtype)
        quadratic = tf.minimum(absolute_error, delta)
        linear = absolute_error - quadratic
        return 0.5 * tf.square(quadratic) + delta * linear

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, y_pred.dtype)
        level_loss = self._elementwise_huber(y_pred - y_true)

        horizon = tf.linspace(
            tf.cast(0.0, y_pred.dtype),
            tf.cast(1.0, y_pred.dtype),
            OUTPUT_HOURS,
        )
        horizon_weights = 1.0 + tf.cast(
            self.late_horizon_weight,
            y_pred.dtype,
        ) * horizon
        weighted_level_loss = tf.reduce_sum(
            level_loss * horizon_weights,
            axis=-1,
        ) / tf.reduce_sum(horizon_weights)

        true_change = y_true[:, 1:] - y_true[:, :-1]
        predicted_change = y_pred[:, 1:] - y_pred[:, :-1]
        change_loss = tf.reduce_mean(
            self._elementwise_huber(predicted_change - true_change),
            axis=-1,
        )
        return weighted_level_loss + tf.cast(
            self.change_weight,
            y_pred.dtype,
        ) * change_loss

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "delta": self.delta,
                "change_weight": self.change_weight,
                "late_horizon_weight": self.late_horizon_weight,
            }
        )
        return config


def make_optimizer(learning_rate=2e-4, weight_decay=1e-4):
    '''Use AdamW+EMA when supported; retain an Adam fallback.'''
    try:
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
            clipnorm=1.0,
            use_ema=True,
            ema_momentum=0.99,
        )
    except (AttributeError, TypeError):
        return tf.keras.optimizers.Adam(
            learning_rate=learning_rate,
            clipnorm=1.0,
        )


In [5]:
def build_model_2026_08_02(
    X_train,
    learning_rate=2e-4,
    weight_decay=1e-4,
):
    validate_training_tensor(X_train, hours=72, feature_count=11)

    normalizer = layers.Normalization(axis=-1, name="feature_normalization")
    normalizer.adapt(X_train.reshape(-1, X_train.shape[-1]))

    inputs = tf.keras.Input(
        shape=(INPUT_HOURS, len(FEATURE_COLUMNS)),
        name="weather_sequence",
    )
    x = normalizer(inputs)

    x = layers.Conv1D(
        48,
        kernel_size=5,
        padding="same",
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="local_pattern_conv",
    )(x)
    x = layers.LayerNormalization(name="conv_normalization")(x)
    x = layers.SpatialDropout1D(
        0.12,
        name="conv_spatial_dropout",
    )(x)

    x = layers.LSTM(
        64,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="history_lstm_1",
    )(x)
    x = layers.LSTM(
        40,
        return_sequences=True,
        dropout=0.15,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="history_lstm_2",
    )(x)

    self_attention = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=20,
        dropout=0.15,
        name="history_self_attention",
    )(x, x)
    x = layers.Add(name="history_attention_residual")([x, self_attention])
    encoded_history = layers.LayerNormalization(
        name="encoded_history",
    )(x)

    average = layers.GlobalAveragePooling1D(
        name="average_context",
    )(encoded_history)
    maximum = layers.GlobalMaxPooling1D(
        name="maximum_context",
    )(encoded_history)
    latest = layers.Cropping1D(
        cropping=(INPUT_HOURS - 1, 0),
        name="latest_context_crop",
    )(encoded_history)
    latest = layers.Reshape((40,), name="latest_context")(latest)

    context = layers.Concatenate(name="combined_context")(
        [average, maximum, latest]
    )
    context = layers.Dense(
        80,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="context_dense",
    )(context)
    context = layers.Dropout(0.25, name="context_dropout")(context)

    horizon_embedding = HorizonEmbedding(
        output_hours=OUTPUT_HOURS,
        embedding_dim=12,
        name="horizon_embedding",
    )(context)
    future_calendar = FutureCalendarFeatures(
        output_hours=OUTPUT_HOURS,
        name="future_calendar",
    )(inputs)
    baselines = TemperatureBaselines(
        output_hours=OUTPUT_HOURS,
        feature_index=TEMPERATURE_FEATURE_INDEX,
        name="temperature_baselines",
    )(inputs)

    forecast_queries = layers.Concatenate(
        name="forecast_query_features",
    )([horizon_embedding, future_calendar, baselines])
    forecast_queries = layers.Dense(
        40,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="forecast_query_projection",
    )(forecast_queries)

    cross_attention = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=20,
        dropout=0.15,
        name="forecast_to_history_attention",
    )(
        query=forecast_queries,
        value=encoded_history,
        key=encoded_history,
    )
    forecast_queries = layers.Add(name="cross_attention_residual")(
        [forecast_queries, cross_attention]
    )
    forecast_queries = layers.LayerNormalization(
        name="cross_attention_normalization",
    )(forecast_queries)

    repeated_context = layers.RepeatVector(
        OUTPUT_HOURS,
        name="repeat_global_context",
    )(context)
    decoder_input = layers.Concatenate(name="decoder_input")(
        [
            forecast_queries,
            repeated_context,
            future_calendar,
            baselines,
        ]
    )
    decoder = layers.GRU(
        48,
        return_sequences=True,
        dropout=0.12,
        kernel_regularizer=regularizers.l2(8e-5),
        recurrent_regularizer=regularizers.l2(4e-5),
        name="forecast_decoder_gru",
    )(decoder_input)
    decoder = layers.Dense(
        40,
        activation="swish",
        kernel_regularizer=regularizers.l2(8e-5),
        name="decoder_dense",
    )(decoder)
    decoder = layers.Dropout(0.12, name="decoder_dropout")(decoder)

    baseline_weights = layers.Dense(
        3,
        activation="softmax",
        name="baseline_weights",
    )(decoder)
    blended_baseline = WeightedBaseline(
        name="blended_baseline",
    )([baselines, baseline_weights])

    correction = layers.Dense(
        1,
        kernel_regularizer=regularizers.l2(5e-5),
        name="temperature_correction",
    )(decoder)
    correction = layers.Reshape(
        (OUTPUT_HOURS,),
        name="temperature_correction_vector",
    )(correction)
    outputs = layers.Add(name="temperature_prediction")(
        [blended_baseline, correction]
    )

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="weather_lstm_cross_attention_24h",
    )
    model.compile(
        optimizer=make_optimizer(learning_rate, weight_decay),
        loss=LevelChangeHuber(
            delta=1.0,
            change_weight=0.25,
            late_horizon_weight=0.30,
        ),
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(name="mae"),
            tf.keras.metrics.RootMeanSquaredError(name="rmse"),
        ],
    )
    return model


## 5. Rebuild and verify every supplied summary

This smoke test uses small synthetic arrays only to adapt each normalization layer and build the graphs. It does not train the models. The assertions compare each graph with the parameter count and output shape in its supplied `model_summary.txt`.

When using real data, pass only the training split to a builder so the normalization statistics do not absorb validation or test information.


In [6]:
rng = np.random.default_rng(21)
demo_24x10 = rng.normal(size=(32, 24, 10)).astype(np.float32)
demo_72x11 = rng.normal(size=(32, 72, 11)).astype(np.float32)

models = {
    "2026-07-01": build_model_2026_07_01(demo_24x10),
    "2026-07-23": build_model_2026_07_23(demo_72x11),
    "2026-07-27": build_model_2026_07_27(demo_72x11),
    "2026-08-02": build_model_2026_08_02(demo_72x11),
}

expected = {
    "2026-07-01": {"output_shape": (None, 1), "params": 33_238},
    "2026-07-23": {"output_shape": (None, 24), "params": 20_287},
    "2026-07-27": {"output_shape": (None, 24), "params": 212_232},
    "2026-08-02": {"output_shape": (None, 24), "params": 100_195},
}

print(f"{'Experiment':<12} {'Output shape':<18} {'Parameters':>12}")
print("-" * 46)
for experiment, model in models.items():
    actual_shape = tuple(model.output_shape)
    actual_params = model.count_params()
    print(f"{experiment:<12} {str(actual_shape):<18} {actual_params:>12,}")
    assert actual_shape == expected[experiment]["output_shape"]
    assert actual_params == expected[experiment]["params"]

print("\nAll four builders match their supplied summaries.")



Experiment   Output shape         Parameters
----------------------------------------------
2026-07-01   (None, 1)                33,238
2026-07-23   (None, 24)               20,287
2026-07-27   (None, 24)              212,232
2026-08-02   (None, 24)              100,195

All four builders match their supplied summaries.


## 6. Choosing and training a model

- Use the early stacked LSTM only when the task truly requires one fixed forecast horizon and the input has the original 10-feature contract.
- Use the July 27 model when a high-capacity previous-day correction model is desired and compute/memory are less constrained.
- Use the August 2 model for the richest 24-hour design: it is less than half the size of the July 27 model, retains direct access to every historical hour, and can adapt its physical baseline at every forecast horizon.

For the 24-hour models, targets must have shape `(samples, 24)`. Keep train/validation/test splits chronological, build the model with the training tensor only, and use `shuffle=False` during fitting.


In [7]:
# Example training pattern (replace the placeholders with real chronological arrays):
#
# model = build_model_2026_08_02(X_train)
# history = model.fit(
#     X_train,
#     y_train,                       # shape: (samples, 24)
#     validation_data=(X_val, y_val),
#     epochs=100,
#     batch_size=64,
#     shuffle=False,
#     callbacks=[
#         tf.keras.callbacks.ReduceLROnPlateau(
#             monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6
#         ),
#         tf.keras.callbacks.EarlyStopping(
#             monitor="val_loss", patience=10, restore_best_weights=True
#         ),
#     ],
# )
